In [1]:
##import modules
import pandas as pd

from dsipts import Categorical,TimeSeries, TTM

print("> Libraries Imported.")

> Libraries Imported.


## Prepare TS

In [2]:
#BEST_DEVICES = ['Z-WAVE_7E','Z-WAVE_8D', 'Z-WAVE_12F', 'Z-WAVE_8F', 'Z-WAVE_7D','Z-WAVE_7F', 'Z-WAVE_4C', 'Z-WAVE_5D']
TIME_COL = "timestamp"
CAT_COLS = ['ora', 'weekend', 'giorno', 'festività']
EXOG_COLS = ['temperature','humidity']
TARGET_COl = ["ElectricWConsumed"] 

#PATH = '/home/davide/Documents/WORKSPACE/inCUBE/data/target.csv'
#PATH = '/home/davide/Documents/WORKSPACE/inCUBE/data/target_noscaled.csv'
PATH = '/home/davide/Documents/WORKSPACE/inCUBE/data/target_withnan.csv'
df = pd.read_csv(PATH, parse_dates=[TIME_COL])
df.rename(columns={TIME_COL: 'time'}, inplace=True)

print(f"> Read: {df.shape}")

> Read: (133919, 16)


In [3]:
df_device = df.loc[df['DEVICE'] == 'Z-WAVE_7E', 
                   ['time'] + CAT_COLS + EXOG_COLS + TARGET_COl].copy()
print(f"> DF size: {df_device.shape}")

> DF size: (3210, 8)


In [5]:
##initizate a timeseries object
ts = TimeSeries('prova')
ts.load_signal(df_device, future_variables=EXOG_COLS, target_variables = TARGET_COl,cat_var = CAT_COLS)
ts

Timeseries named prova of length 3210.
 Categorical variable: ['ora', 'weekend', 'giorno', 'festività'],
 Future variables: ['temperature', 'humidity'],
 Past variables: ['ElectricWConsumed'],
 Target variables: ['ElectricWConsumed']
 With no group

## Set Model

In [6]:
past_steps = 100
future_steps = 20

In [9]:
ts.dataset

,time,ora,weekend,giorno,festività,temperature,humidity,ElectricWConsumed
0,2024-06-06 15:00:00,15,0,3,0,27.612830,55.578057,90295.774
1,2024-06-06 16:00:00,16,0,3,0,27.273167,54.175533,163387.916
2,2024-06-06 17:00:00,17,0,3,0,27.284183,54.681817,122336.383
3,2024-06-06 18:00:00,18,0,3,0,26.942233,53.665500,3006.737
4,2024-06-06 19:00:00,19,0,3,0,26.530183,53.503383,2921.996
...,...,...,...,...,...,...,...,...
3205,2024-10-18 04:00:00,4,0,4,0,22.319083,64.775717,2813.377
3206,2024-10-18 05:00:00,5,0,4,0,22.358288,65.112576,2927.959
3207,2024-10-18 06:00:00,6,0,4,0,22.256050,64.512550,2842.803
3208,2024-10-18 07:00:00,7,0,4,0,22.305433,64.790867,2915.538


In [19]:
config = dict(model_configs =dict(
                                model_path="ibm-granite/granite-timeseries-ttm-r2",
                                num_input_channels=len(ts.dataset.columns),  # exog: number of input channels
                                decoder_mode="mix_channel",  # exog:  set to mix_channel for mixing channels in history IF REMOVED GIVES THE BEST RESULTS
                                #mode='mix_channel', # NOTE: ADDED THIS  IF REMOVED GIVES THE BEST RESULTS
                                prediction_channel_indices=[ts.dataset.columns.get_loc(c) for c in ts.target_variables],
                                exogenous_channel_indices=[ts.dataset.columns.get_loc(c) for c in ts.cat_var + EXOG_COLS],
                                past_steps=past_steps,
                                future_steps=future_steps,
                                freq_prefix_tuning=False,
                                freq='1h',
                                prefer_l1_loss=False,
                                prefer_longer_context=True,
                                fcm_context_length=1,  # exog: indicates lag length to use in the exog fusion. for Ex. if today sales can get affected by discount on +/- 2 days, mention 2
                                fcm_use_mixer=True,  # exog: Try true (1st option) or false
                                fcm_mix_layers=2,  # exog: Number of layers for exog mixing
                                enable_forecast_channel_mixing=True,  # exog: set true for exog mixing
                                fcm_prepend_past=True,  # exog: set true to include lag from history during exog infusion.
                                # Can also provide TTM Config args
                                embs = [ts.dataset[c].nunique() for c in ts.cat_var],
                                quantiles=[0.1,0.5,0.9],
                                persistence_weight= 0.010,
                                loss_type= 'l1',
                                remove_last= True,
                                optim= 'torch.optim.Adam',
                                activation= 'torch.nn.GELU', 
                                verbose = True,
                                out_channels = len(ts.target_variables)),
                scheduler_config = dict(gamma=0.1,step_size=100),
                optim_config = dict(lr = 0.0005,weight_decay=0.01))
model_sum = TTM(**config['model_configs'],optim_config = config['optim_config'],scheduler_config =config['scheduler_config'] )
ts.set_model(model_sum,config=config )


TypeError: Object of type QuantileLossMO is not JSON serializable